In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable

# Read validated incremental Silver tables
sales_df = spark.read.table("dbo.silver_sales_incremental")
customers_df = spark.read.table("dbo.silver_customers_incremental")
products_df = spark.read.table("dbo.silver_products_incremental")

# Count source records
sales_count = sales_df.count()
customers_count = customers_df.count()
products_count = products_df.count()

print("=== INCREMENTAL GOLD SOURCE VALIDATION ===")
print("Silver Sales:", sales_count)
print("Silver Customers:", customers_count)
print("Silver Products:", products_count)

# Validate that required columns exist
required_columns = {
    "Sales": (
        sales_df,
        {"sale_id", "customer_id", "product_id",
         "sale_date", "quantity", "unit_price"}
    ),
    "Customers": (
        customers_df,
        {"customer_id", "customer_name", "is_deleted"}
    ),
    "Products": (
        products_df,
        {"product_id", "product_name", "category", "is_deleted"}
    )
}

for table_name, (df, required) in required_columns.items():
    missing = required - set(df.columns)
    assert not missing, f"{table_name} missing columns: {missing}"

assert sales_count > 0, "Silver Sales is empty"
assert customers_count > 0, "Silver Customers is empty"
assert products_count > 0, "Silver Products is empty"

print("GOLD SOURCE VALIDATION PASSED")


In [ ]:

# Keep only active customers and products
active_customers_df = customers_df.filter(
    F.col("is_deleted") == False
).select(
    "customer_id",
    "customer_name"
)

active_products_df = products_df.filter(
    F.col("is_deleted") == False
).select(
    "product_id",
    "product_name",
    "category"
)

# Join Sales with active Customers
sales_customers_df = sales_df.join(
    active_customers_df,
    on="customer_id",
    how="inner"
)

# Join the result with active Products
gold_joined_df = sales_customers_df.join(
    active_products_df,
    on="product_id",
    how="inner"
)

# Validate that no valid Silver Sales were lost
joined_count = gold_joined_df.count()

print("=== GOLD JOIN VALIDATION ===")
print("Silver Sales:", sales_count)
print("Gold Joined Sales:", joined_count)

assert joined_count == sales_count, (
    f"Gold join mismatch: Silver={sales_count}, "
    f"Gold={joined_count}"
)

assert (
    gold_joined_df.select("sale_id").distinct().count()
    == joined_count
), "Duplicate sale_id detected after Gold joins"

print("GOLD JOIN VALIDATION PASSED")

gold_joined_df.select(
    "sale_id",
    "customer_name",
    "product_name",
    "category",
    "quantity",
    "unit_price"
).show(5, truncate=False)


In [ ]:

# Calculate revenue using decimal precision
gold_revenue_df = gold_joined_df.withColumn(
    "revenue",
    (
        F.col("quantity") *
        F.col("unit_price")
    ).cast(DecimalType(18, 2))
)

# Validate revenue values
invalid_revenue_count = gold_revenue_df.filter(
    F.col("revenue").isNull() |
    (F.col("revenue") <= 0)
).count()

assert invalid_revenue_count == 0, (
    f"Invalid revenue records: {invalid_revenue_count}"
)

# Calculate overall revenue
total_revenue = gold_revenue_df.agg(
    F.sum("revenue").alias("total_revenue")
).first()["total_revenue"]

print("=== GOLD REVENUE VALIDATION ===")
print("Total Sales:", gold_revenue_df.count())
print("Invalid Revenue Records:", invalid_revenue_count)
print("Total Revenue:", total_revenue)

print("GOLD REVENUE VALIDATION PASSED")

# Display sample sales with calculated revenue
gold_revenue_df.select(
    "sale_id",
    "customer_name",
    "product_name",
    "quantity",
    "unit_price",
    "revenue"
).show(5, truncate=False)


In [ ]:

# Calculate total revenue by product category
category_revenue_df = gold_revenue_df.groupBy(
    "category"
).agg(
    F.sum("revenue").cast(
        DecimalType(18, 2)
    ).alias("total_revenue")
)

# Sort categories by revenue
category_sorted_df = category_revenue_df.orderBy(
    F.col("total_revenue").desc()
)

# Validate aggregated revenue
category_total = category_revenue_df.agg(
    F.sum("total_revenue").alias("total")
).first()["total"]

category_count = category_revenue_df.count()

print("=== GOLD CATEGORY REVENUE VALIDATION ===")
print("Total Categories:", category_count)
print("Category Revenue Total:", category_total)
print("Overall Sales Revenue:", total_revenue)

assert category_total == total_revenue, (
    "Category revenue does not match overall revenue"
)

assert category_count > 0, "No categories found"

print("GOLD CATEGORY REVENUE VALIDATION PASSED")

category_sorted_df.show(truncate=False)


In [ ]:

gold_category_table = "dbo.gold_category_revenue_incremental"

# Create table on first run
if not spark.catalog.tableExists(gold_category_table):

    category_revenue_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(gold_category_table)

    print("Gold Category Revenue table created successfully!")

else:
    # Synchronize existing table with current aggregates
    target = DeltaTable.forName(
        spark,
        gold_category_table
    )

    (
        target.alias("target")
        .merge(
            category_revenue_df.alias("source"),
            "target.category = source.category"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )

    print("Gold Category Revenue MERGE completed!")

# Validate persisted table
saved_category_df = spark.read.table(gold_category_table)

saved_category_total = saved_category_df.agg(
    F.sum("total_revenue").alias("total")
).first()["total"]

assert saved_category_df.count() == category_count, (
    "Saved category count mismatch"
)

assert saved_category_total == total_revenue, (
    "Saved category revenue mismatch"
)

print("=== GOLD CATEGORY TABLE VALIDATION ===")
print("Saved Categories:", saved_category_df.count())
print("Saved Revenue:", saved_category_total)
print("GOLD CATEGORY TABLE VALIDATION PASSED")

saved_category_df.orderBy(
    F.col("total_revenue").desc()
).show(truncate=False)


In [ ]:

# Calculate total quantity and revenue by product
product_revenue_df = gold_revenue_df.groupBy(
    "product_id",
    "product_name"
).agg(
    F.sum("quantity").alias("total_quantity"),
    F.sum("revenue").cast(
        DecimalType(18, 2)
    ).alias("total_revenue")
)

# Sort products by highest revenue
product_sorted_df = product_revenue_df.orderBy(
    F.col("total_revenue").desc()
)

# Validate product revenue
product_total = product_revenue_df.agg(
    F.sum("total_revenue").alias("total")
).first()["total"]

product_count = product_revenue_df.count()

# Check product IDs are unique
duplicate_product_count = (
    product_revenue_df.groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== GOLD PRODUCT REVENUE VALIDATION ===")
print("Total Products:", product_count)
print("Product Revenue Total:", product_total)
print("Overall Sales Revenue:", total_revenue)
print("Duplicate Product IDs:", duplicate_product_count)

assert product_total == total_revenue, (
    "Product revenue does not match overall revenue"
)

assert duplicate_product_count == 0, (
    "Duplicate product IDs detected"
)

assert product_count > 0, "No products found"

print("GOLD PRODUCT REVENUE VALIDATION PASSED")

product_sorted_df.show(truncate=False)


In [ ]:

gold_product_table = "dbo.gold_product_revenue_incremental"

# Create table on the first run
if not spark.catalog.tableExists(gold_product_table):

    product_revenue_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(gold_product_table)

    print("Gold Product Revenue table created successfully!")

else:
    # Synchronize current product aggregates
    target = DeltaTable.forName(
        spark,
        gold_product_table
    )

    (
        target.alias("target")
        .merge(
            product_revenue_df.alias("source"),
            "target.product_id = source.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )

    print("Gold Product Revenue MERGE completed!")

# Validate persisted Gold table
saved_product_df = spark.read.table(gold_product_table)

saved_product_count = saved_product_df.count()

saved_product_total = saved_product_df.agg(
    F.sum("total_revenue").alias("total")
).first()["total"]

saved_product_duplicates = (
    saved_product_df.groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== GOLD PRODUCT TABLE VALIDATION ===")
print("Saved Products:", saved_product_count)
print("Saved Revenue:", saved_product_total)
print("Duplicate Product IDs:", saved_product_duplicates)

assert saved_product_count == product_count, (
    "Saved product count mismatch"
)

assert saved_product_total == total_revenue, (
    "Saved product revenue mismatch"
)

assert saved_product_duplicates == 0, (
    "Duplicate product IDs in Gold table"
)

print("GOLD PRODUCT TABLE VALIDATION PASSED")

saved_product_df.orderBy(
    F.col("total_revenue").desc()
).show(truncate=False)


In [ ]:

# Calculate total revenue by customer
customer_revenue_df = gold_revenue_df.groupBy(
    "customer_id",
    "customer_name"
).agg(
    F.sum("revenue").cast(
        DecimalType(18, 2)
    ).alias("total_revenue")
)

# Sort customers by highest revenue
customer_sorted_df = customer_revenue_df.orderBy(
    F.col("total_revenue").desc()
)

# Validate customer revenue
customer_total = customer_revenue_df.agg(
    F.sum("total_revenue").alias("total")
).first()["total"]

customer_count = customer_revenue_df.count()

# Check duplicate customer IDs
duplicate_customer_count = (
    customer_revenue_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== GOLD CUSTOMER REVENUE VALIDATION ===")
print("Total Customers:", customer_count)
print("Customer Revenue Total:", customer_total)
print("Overall Sales Revenue:", total_revenue)
print("Duplicate Customer IDs:", duplicate_customer_count)

assert customer_total == total_revenue, (
    "Customer revenue does not match overall revenue"
)

assert duplicate_customer_count == 0, (
    "Duplicate customer IDs detected"
)

assert customer_count > 0, "No customers found"

print("GOLD CUSTOMER REVENUE VALIDATION PASSED")

customer_sorted_df.show(truncate=False)


In [ ]:

gold_customer_table = "dbo.gold_customer_revenue_incremental"

# Create table on the first run
if not spark.catalog.tableExists(gold_customer_table):

    customer_revenue_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(gold_customer_table)

    print("Gold Customer Revenue table created successfully!")

else:
    # Synchronize current customer aggregates
    target = DeltaTable.forName(
        spark,
        gold_customer_table
    )

    (
        target.alias("target")
        .merge(
            customer_revenue_df.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )

    print("Gold Customer Revenue MERGE completed!")

# Validate persisted Gold table
saved_customer_df = spark.read.table(gold_customer_table)

saved_customer_count = saved_customer_df.count()

saved_customer_total = saved_customer_df.agg(
    F.sum("total_revenue").alias("total")
).first()["total"]

saved_customer_duplicates = (
    saved_customer_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== GOLD CUSTOMER TABLE VALIDATION ===")
print("Saved Customers:", saved_customer_count)
print("Saved Revenue:", saved_customer_total)
print("Duplicate Customer IDs:", saved_customer_duplicates)

assert saved_customer_count == customer_count, (
    "Saved customer count mismatch"
)

assert saved_customer_total == total_revenue, (
    "Saved customer revenue mismatch"
)

assert saved_customer_duplicates == 0, (
    "Duplicate customer IDs in Gold table"
)

print("GOLD CUSTOMER TABLE VALIDATION PASSED")

saved_customer_df.orderBy(
    F.col("total_revenue").desc()
).show(truncate=False)


In [ ]:

# Read persisted Gold tables
gold_category_df = spark.read.table(
    "dbo.gold_category_revenue_incremental"
)

gold_product_df = spark.read.table(
    "dbo.gold_product_revenue_incremental"
)

gold_customer_df = spark.read.table(
    "dbo.gold_customer_revenue_incremental"
)

# Calculate totals from saved tables
category_total = gold_category_df.agg(
    F.sum("total_revenue")
).first()[0]

product_total = gold_product_df.agg(
    F.sum("total_revenue")
).first()[0]

customer_total = gold_customer_df.agg(
    F.sum("total_revenue")
).first()[0]

# Calculate source revenue independently
silver_total = gold_revenue_df.agg(
    F.sum("revenue")
).first()[0]

# Verify quantity reconciliation
silver_quantity = gold_revenue_df.agg(
    F.sum("quantity")
).first()[0]

product_quantity = gold_product_df.agg(
    F.sum("total_quantity")
).first()[0]

# Verify business-key uniqueness
category_keys = gold_category_df.select(
    "category"
).distinct().count()

product_keys = gold_product_df.select(
    "product_id"
).distinct().count()

customer_keys = gold_customer_df.select(
    "customer_id"
).distinct().count()

print("=== FINAL INCREMENTAL GOLD VALIDATION ===")
print("Silver Sales:", sales_df.count())
print("Gold Joined Sales:", gold_revenue_df.count())
print("Silver Revenue:", silver_total)
print("Category Revenue:", category_total)
print("Product Revenue:", product_total)
print("Customer Revenue:", customer_total)
print("Silver Quantity:", silver_quantity)
print("Gold Product Quantity:", product_quantity)

assert sales_df.count() == gold_revenue_df.count(), (
    "Sales count mismatch"
)

assert silver_total == category_total == product_total == customer_total, (
    "Gold revenue reconciliation failed"
)

assert silver_quantity == product_quantity, (
    "Gold product quantity reconciliation failed"
)

assert category_keys == gold_category_df.count(), (
    "Duplicate Gold category keys"
)

assert product_keys == gold_product_df.count(), (
    "Duplicate Gold product keys"
)

assert customer_keys == gold_customer_df.count(), (
    "Duplicate Gold customer keys"
)

print("FINAL INCREMENTAL GOLD VALIDATION PASSED")


In [ ]:

from pyspark.sql import functions as F

gold_tables = {
    "Category": "dbo.gold_category_revenue_incremental",
    "Product": "dbo.gold_product_revenue_incremental",
    "Customer": "dbo.gold_customer_revenue_incremental"
}

print("=== POST-PIPELINE GOLD VALIDATION ===")

totals = {}

for name, table_name in gold_tables.items():
    df = spark.read.table(table_name)

    count = df.count()
    revenue = df.agg(
        F.sum("total_revenue")
    ).first()[0]

    totals[name] = revenue

    print(f"{name} Records: {count}")
    print(f"{name} Revenue: {revenue}")

assert len(set(totals.values())) == 1, (
    "Gold revenue totals do not match"
)

silver_sales = spark.read.table(
    "dbo.silver_sales_incremental"
)

silver_revenue = silver_sales.agg(
    F.sum(
        F.col("quantity") * F.col("unit_price")
    ).alias("revenue")
).first()["revenue"]

print("Silver Revenue:", silver_revenue)

assert all(
    revenue == silver_revenue
    for revenue in totals.values()
), "Gold and Silver revenue mismatch"

print("POST-PIPELINE GOLD VALIDATION PASSED")
